<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_03_matmul_variants.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-03: Matrix Multiplication Variants

**Difficulty**: ⬜ Warm-up

**Objective**: Know ALL the ways to multiply matrices in PyTorch — you'll use every one of these.

| Method | Use Case |
|--------|----------|
| `@` / `torch.matmul` | General purpose, handles batches |
| `torch.bmm` | Strictly batched (3D only) |
| `torch.einsum` | Any contraction — the Swiss army knife |
| `F.linear` | Linear layer: `x @ W.T + b` |

In [1]:
import torch
import torch.nn.functional as F

## Task 1: Basic 2D Matmul

In [2]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)

# TODO: Compute C = A @ B three ways
c1 = A @ B  # using @
c2 = torch.matmul(A, B)  # using torch.matmul
c3 = torch.einsum('ij,jk->ik', A, B)  # using torch.einsum (hint: 'ij,jk->ik')

assert c1.shape == (3, 5)
assert torch.allclose(c1, c2)
assert torch.allclose(c1, c3)
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Batched Matmul

This is what attention computes: `(B, H, N, D) @ (B, H, D, N)` → `(B, H, N, N)`

In [3]:
B, H, N, D = 2, 8, 10, 64
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)

# TODO: Compute attention scores Q @ K^T → shape (B, H, N, N)
scores_1 = Q @ K.transpose(3, 2)  # using @ or matmul (hint: K needs transpose on last two dims)
scores_2 = torch.einsum('bhnd,bhmd->bhnm', Q, K)  # using einsum (hint: 'bhnd,bhmd->bhnm')

assert scores_1.shape == (B, H, N, N)
assert torch.allclose(scores_1, scores_2, atol=1e-5)
print('✅ Task 2 passed!')

✅ Task 2 passed!


## Task 3: F.linear — The Linear Layer Shortcut

In [4]:
x = torch.randn(2, 10, 64)  # (batch, seq, features)
W = torch.randn(128, 64)     # (out_features, in_features)
b = torch.randn(128)         # (out_features,)

# TODO: Compute y = x @ W.T + b two ways
y1 = x @ W.transpose(1, 0) + b  # manual: x @ W.T + b
y2 = F.linear(x, W, b)  # using F.linear(x, W, b)

assert y1.shape == (2, 10, 128)
assert torch.allclose(y1, y2, atol=1e-5)
print('✅ Task 3 passed!')

✅ Task 3 passed!


## Task 4: Einsum for Everything

Translate each operation to einsum:

In [6]:
# 4a: Dot product of two vectors
a = torch.randn(5)
b = torch.randn(5)
dot = torch.einsum('i,i->', a, b)  # einsum: 'i,i->'
assert torch.allclose(dot, (a * b).sum())

# 4b: Outer product
outer = torch.einsum('i,j->ij', a, b)  # einsum: 'i,j->ij'
assert outer.shape == (5, 5)

# 4c: Batch trace: given (B, N, N), compute trace of each matrix
M = torch.randn(3, 4, 4)
traces = torch.einsum('bii->b', M)  # einsum: 'bii->b'
assert traces.shape == (3,)

# 4d: Bilinear: x^T A y where x:(B,D), A:(D,D), y:(B,D) → (B,)
x = torch.randn(4, 8)
A = torch.randn(8, 8)
y = torch.randn(4, 8)
bilinear = torch.einsum('bi,ij,bj->b', x, A, y)  # einsum: 'bi,ij,bj->b'
assert bilinear.shape == (4,)

print('✅ Task 4 passed!')

✅ Task 4 passed!
